In [ ]:
import csv
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore') # Does not work for this kind of warning
pd.set_option('display.max_rows', None)


In [ ]:
# Compensate for deficits
def generate_dynamic_deficit_schedule(start_date_str, total_weeks=26):
    workers = [
        "Dr. Alice Smith", "Nurse Bob Jones", "Dr. Charlie Brown", "Nurse Diana Prince",
        "Dr. Evan Wright", "Nurse Fiona Gallagher", "Dr. George Clark", "Nurse Hannah Abbott", "midwife Cezanne"
    ]
    
    # Public Holidays (Each drops that individual week's target base down to 32 hours)
    PUBLIC_HOLIDAYS = {
        "2026-10-12", "2026-11-11", "2026-11-26", "2026-12-25", "2027-01-01", "2027-01-19"
    }

    staff_vacations = {
        "Dr. Alice Smith": {"2026-09-24", "2026-09-25"},
        "Nurse Bob Jones": set(),
        "Dr. Charlie Brown": {"2026-10-01", "2026-10-02"},
        "Nurse Diana Prince": set(),
        "Dr. Evan Wright": {"2026-12-24", "2026-12-26"}, 
        "Nurse Fiona Gallagher": set(),
        "Dr. George Clark": set(),
        "Nurse Hannah Abbott": set(),
        "midwife Cezanne": set()
    }
    
    # Global tracking counters over the 6 months
    global_counts = {w: {'Day_Shifts': 0, 'Night_Shifts': 0, 'Full_Clinic': 0, 'Half_Clinic': 0, 'Total_Hours': 0} for w in workers}
    
    # NEW: Track running deficit hours for each staff member
    deficit_hours = {w: 0.0 for w in workers}
    
    schedule_data = []
    last_night_worker = None
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")

    for week_idx in range(total_weeks):
        week_start_date = start_date + timedelta(weeks=week_idx)
        weekly_hospital_counts = {w: 0 for w in workers}
        week_days_manifest = {}
        
        # Check if the current week contains a public holiday
        has_holiday_this_week = False
        for day_offset in range(7):
            if (week_start_date + timedelta(days=day_offset)).strftime("%Y-%m-%d") in PUBLIC_HOLIDAYS:
                has_holiday_this_week = True
        
        # Determine base target for the week
        weekly_base_target = 32 if has_holiday_this_week else 40

        # --- PHASE 1: Assign 12-Hour Hospital Shifts (Mon - Sun) ---
        for day_offset in range(7):
            current_date = week_start_date + timedelta(days=day_offset)
            date_str = current_date.strftime("%Y-%m-%d")
            day_name = current_date.strftime("%A")
            
            week_days_manifest[date_str] = {
                'day_name': day_name, 'day_hospital': None, 'night_hospital': None, 
                'full_clinic_staff': [], 'half_clinic_staff': []
            }
            
            active_today = [w for w in workers if date_str not in staff_vacations[w]]
            
            # Day Shift
            available_for_day = [w for w in active_today if w != last_night_worker and weekly_hospital_counts[w] < 3]
            # Equity Sort: Prioritize by current local week load, then total historical hours logged
            available_for_day.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Day_Shifts'], global_counts[w]['Total_Hours']))
            assigned_day_worker = available_for_day[0]
            
            week_days_manifest[date_str]['day_hospital'] = assigned_day_worker
            weekly_hospital_counts[assigned_day_worker] += 1
            global_counts[assigned_day_worker]['Day_Shifts'] += 1
            global_counts[assigned_day_worker]['Total_Hours'] += 12
            
            # Night Shift
            available_for_night = [w for w in active_today if w != assigned_day_worker and w != last_night_worker and weekly_hospital_counts[w] < 3]
            available_for_night.sort(key=lambda w: (weekly_hospital_counts[w], global_counts[w]['Night_Shifts'], global_counts[w]['Total_Hours']))
            assigned_night_worker = available_for_night[0]
            
            week_days_manifest[date_str]['night_hospital'] = assigned_night_worker
            weekly_hospital_counts[assigned_night_worker] += 1
            global_counts[assigned_night_worker]['Night_Shifts'] += 1
            global_counts[assigned_night_worker]['Total_Hours'] += 12
            
            last_night_worker = assigned_night_worker

        # --- PHASE 2: Dynamic Clinic Assignment Incorporating Deficit Carryover ---
        for worker in workers:
            hospital_hours = weekly_hospital_counts[worker] * 12
            
            # Personal target for this week = base target (32 or 40) + any accumulated deficit hours owed
            personal_target = weekly_base_target + deficit_hours[worker]
            needed_clinic_hours = max(0, personal_target - hospital_hours)
            
            # Enforce an absolute maximum ceiling of 60 total hours per week for safety
            if hospital_hours + needed_clinic_hours > 60:
                needed_clinic_hours = 60 - hospital_hours
                
            # Break the required clinic hours down into standard 8-hour and 4-hour components
            req_full = int(needed_clinic_hours // 8)
            req_half = int((needed_clinic_hours % 8) // 4)
            
            full_assigned = 0
            half_assigned = 0
            
            # Enforce weekday allocation loops
            for date_str, day_data in week_days_manifest.items():
                if full_assigned == req_full and half_assigned == req_half:
                    break
                
                if day_data['day_name'] in ['Saturday', 'Sunday'] or date_str in PUBLIC_HOLIDAYS or date_str in staff_vacations[worker]:
                    continue
                
                if day_data['day_hospital'] != worker and day_data['night_hospital'] != worker:
                    # Post-night shift rest check
                    prev_date_str = (datetime.strptime(date_str, "%Y-%m-%d") - timedelta(days=1)).strftime("%Y-%m-%d")
                    was_resting = prev_date_str in week_days_manifest and week_days_manifest[prev_date_str]['night_hospital'] == worker
                    
                    if not was_resting:
                        if full_assigned < req_full:
                            day_data['full_clinic_staff'].append(worker)
                            full_assigned += 1
                            global_counts[worker]['Full_Clinic'] += 1
                            global_counts[worker]['Total_Hours'] += 8
                        elif half_assigned < req_half:
                            day_data['half_clinic_staff'].append(worker)
                            half_assigned += 1
                            global_counts[worker]['Half_Clinic'] += 1
                            global_counts[worker]['Total_Hours'] += 4

            # Calculate actual hours completed this week across hospital + clinic
            hours_worked_this_week = hospital_hours + (full_assigned * 8) + (half_assigned * 4)
            
            # Recalculate deficit carryover balance for next week's loop iteration
            deficit_hours[worker] = max(0, personal_target - hours_worked_this_week)

        # --- PHASE 3: Compile Manifest ---
        for date_str, day_data in sorted(week_days_manifest.items()):
            is_holiday = date_str in PUBLIC_HOLIDAYS
            schedule_data.append({
                'Date': date_str,
                'Day of Week': day_data['day_name'] + (" 🎉 [HOLIDAY]" if is_holiday else ""),
                'Day Shift (12hr)': day_data['day_hospital'],
                'Night Shift (12hr)': day_data['night_hospital'],
                'Full Clinic (8hr)': "🏥 CLOSED" if is_holiday else (", ".join(day_data['full_clinic_staff']) if day_data['full_clinic_staff'] else "None"),
                'Half Clinic (4hr)': "🏥 CLOSED" if is_holiday else (", ".join(day_data['half_clinic_staff']) if day_data['half_clinic_staff'] else "None")
            })
    tab = pd.DataFrame(schedule_data)
    # 4. Export to clean structural spreadsheet
    csv_filename = "hospital_holiday_40hr_schedule.csv"
    fields = ['Date', 'Day of Week', 'Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']
    with open(csv_filename, mode='w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(schedule_data)
        
    print(f"Schedule exported successfully with running balance tracking features to {csv_filename}!\n")
    print("6-Month Cumulative Totals (Vacations Deducted, Equity Maintained):")
    print(f"{'Worker Name':<22} | {'Day Shifts':<10} | {'Night Calls':<11} | {'Full Clinic':<11} | {'Half Clinic':<11} | {'Total Hours':<12}")
    print("-" * 92)
    for worker, counts in sorted(global_counts.items()):
        print(f"{worker:<22} | {counts['Day_Shifts']:<10} | {counts['Night_Shifts']:<11} | {counts['Full_Clinic']:<11} | {counts['Half_Clinic']:<11} | {counts['Total_Hours']:<12} hrs")
    return tab
dat = generate_dynamic_deficit_schedule("2026-09-20")


In [ ]:
dat.columns = ['date','day_of_week','day_shift','night_shift','full_clinic','half_clinic']
dat['date'] = pd.to_datetime(dat['date'])
dat['week_no'] = (dat['date'] + pd.Timedelta(days=1)).dt.isocalendar().week
#dat['week_no'] = dat['date'].dt.isocalendar().week


In [ ]:
dat[(dat.day_of_week == 'Saturday') | (dat.day_of_week == 'Sunday') ].day_shift.value_counts()